<a href="https://colab.research.google.com/github/NotfromEQ/Step_Up/blob/Training_Model_Playground/Web_for_handwriting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. ติดตั้ง Gradio (ถ้ายังไม่เคยติดตั้งใน Colab)
!pip install gradio torch torchvision

import gradio as gr
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
import numpy as np

# 2. เตรียม Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 3. นิยามโครงสร้างโมเดล CNNNet (ต้องเหมือนตอนเทรนเป๊ะ!)
class CNNNet(nn.Module):
    def __init__(self):
        super(CNNNet, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.fc = nn.Linear(32 * 7 * 7, 10)

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

# 4. โหลดโมเดลที่เทรนไว้ (ไฟล์ my_super_cnn.pth ต้องอยู่ในโฟลเดอร์เดียวกัน!)
model = CNNNet().to(device)
try:
    model.load_state_dict(torch.load('my_super_cnn.pth', map_location=device))
    model.eval() # ตั้งค่าเป็นโหมดประเมินผล (ไม่ต้องเทรนแล้ว)
    print("โมเดลโหลดสำเร็จและพร้อมใช้งาน!")
except FileNotFoundError:
    print("ไม่พบไฟล์ 'my_super_cnn.pth' กรุณาตรวจสอบว่าอยู่ในโฟลเดอร์เดียวกับโค้ดนี้")
    print("รันคำสั่ง torch.save(model.state_dict(), 'my_super_cnn.pth') อีกครั้งหากยังไม่เซฟ")
    # หากไม่พบไฟล์ จะโหลดโมเดลตัวอย่างแทน
    # หากยังไม่ได้เซฟไฟล์ ให้รัน Cell เทรนก่อน
    # หรือโหลดโมเดลตัวอย่างจาก Hugging Face (ถ้ามี)
    # เช่น model = torch.hub.load('pytorch/vision:v0.10.0', 'resnet18', pretrained=True)
    exit() # หยุดการทำงานหากไม่พบโมเดล


# 5. กำหนดฟังก์ชันสำหรับประมวลผลรูปภาพที่รับจากเว็บ UI
def classify_digit(image):
    if image is None:
        return "กรุณาวาดตัวเลข"

    # แปลงรูปภาพให้เป็นขาวดำ (Grayscale)
    image = Image.fromarray(image).convert("L")

    # ปรับขนาดรูปภาพให้เป็น 28x28 พิกเซล
    image = image.resize((28, 28))

    # แปลงรูปภาพเป็น Tensor และ Normalize เหมือนตอนเทรน
    # MNIST ใช้ค่า Normalize คือ (0.1307,) สำหรับ Mean และ (0.3081,) สำหรับ Std
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])
    input_tensor = transform(image).unsqueeze(0).to(device) # เพิ่มมิติ Batch

    with torch.no_grad():
        output = model(input_tensor)
        probabilities = torch.nn.functional.softmax(output[0], dim=0) # แปลงเป็น %
        predicted_class_idx = torch.argmax(probabilities).item()

    # สร้างข้อความผลลัพธ์พร้อมความน่าจะเป็น
    return f"AI ทายว่า: {predicted_class_idx} (ความมั่นใจ: {probabilities[predicted_class_idx].item()*100:.2f}%)"

# 6. สร้าง Gradio Interface
iface = gr.Interface(
    fn=classify_digit,
    inputs=gr.Image(shape=(280, 280), image_mode="L", invert_colors=True, source="canvas", tool="sketch"),
    outputs="label",
    live=True,
    title="ทายเลข MNIST ด้วย AI ของอิคคิว",
    description="วาดตัวเลข 0-9 ลงบนกระดานด้านซ้าย แล้ว AI จะทายว่าเป็นเลขอะไร"
)

# 7. เปิดตัวเว็บ UI
iface.launch(share=True)